In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
import re
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from urllib.parse import urlparse
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class ModelConfig:
    """Configuration for model paths and hyperparameters"""
    url_model_path: str = r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl"
    text_model_path: str = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
    gating_network_path: str = "gating_network.pth"
    max_text_length: int = 128
    phishing_threshold: float = 0.6
    confidence_threshold: float = 0.8

# ============================================================================
# ENHANCED GATING NETWORK
# ============================================================================

class EnhancedGatingNetwork(nn.Module):
    """Improved gating network with better feature processing"""
    
    def __init__(self, input_size=10, hidden_size=128, num_experts=2, dropout=0.3):
        super(EnhancedGatingNetwork, self).__init__()
        
        # Feature processor with residual connections
        self.feature_processor = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )
        
        # Gating mechanism
        self.gate = nn.Sequential(
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.ReLU(),
            nn.Linear(hidden_size // 4, num_experts),
        )
        
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        features = self.feature_processor(x)
        gate_output = self.gate(features)
        weights = self.softmax(gate_output)
        return weights

# ============================================================================
# URL FEATURES
# ============================================================================

class URLFeatures(BaseEstimator, TransformerMixin):
    """Feature extractor for URL-based phishing detection"""
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, urls):
        urls = np.array(urls).reshape(-1)
        feats = np.array([self._extract_features(u) for u in urls])
        return csr_matrix(feats)
    
    def _extract_features(self, url: str) -> List[float]:
        """Extract URL features"""
        if not url or pd.isna(url):
            return [0] * 8
        
        try:
            return [
                len(url),
                url.count('-'),
                url.count('@'),
                url.count('?'),
                url.count('='),
                url.count('.'),
                int(url.startswith("https")),
                int(url.count("//") > 1),
            ]
        except Exception:
            return [0] * 8

# ============================================================================
# ADVANCED FEATURE EXTRACTOR
# ============================================================================

class AdvancedFeatureExtractor:
    """Enhanced feature extraction with balanced input representation"""
    
    PHRASE_DICT = {
        'urgent': 0.6, 'immediately': 0.6, 'act now': 0.6, 'limited time': 0.4,
        'expires today': 0.6, 'last chance': 0.6,
        'verify account': 0.7, 'suspended': 0.6, 'confirm your': 0.6,
        'update account': 0.6, 'security alert': 0.7, 'unusual activity': 0.6,
        'verify identity': 0.7, 'locked': 0.6, 'restricted': 0.5,
        'congratulations': 0.5, 'winner': 0.7, 'claim': 0.6, 'prize': 0.5,
        'free money': 0.6, 'cash prize': 0.6, 'refund': 0.4, 'bonus': 0.3,
        'click here': 0.5, 'click now': 0.5, 'download': 0.3, 'open attachment': 0.4,
        'confirm': 0.3, 'validate': 0.4, 'reactivate': 0.5,
        'free': 0.5, 'offer': 0.2, 'deal': 0.2, 'discount': 0.2,
    }
    
    SUSPICIOUS_TLDS = ['.tk', '.ml', '.ga', '.cf', '.xyz', '.top', '.click', '.link']
    
    @staticmethod
    def preprocess_text(text: str) -> str:
        """Enhanced text preprocessing"""
        if pd.isna(text) or text == "":
            return ""
        
        text = str(text)
        text = re.sub(r'http\S+|www\.\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'([!?.]){2,}', r'\1', text)
        return text
    
    @classmethod
    def calculate_phrase_score(cls, text: str) -> float:
        """Calculate weighted phishing phrase score"""
        if not text:
            return 0.0
        
        text_lower = text.lower()
        score = 0.0
        
        for phrase, weight in cls.PHRASE_DICT.items():
            if phrase in text_lower:
                score += weight
        
        # Normalize by number of unique words to prevent domination
        unique_words = len(set(text_lower.split()))
        if unique_words > 0:
            score = score / (1 + np.log(unique_words))
        
        return min(score, 1.0)
    
    @classmethod
    def extract_url_quality_features(cls, url: str) -> List[float]:
        """Extract URL quality indicators for gating"""
        if not url or pd.isna(url) or url == "":
            return [0.0, 0.0, 0.0, 0.0]
        
        try:
            url_lower = url.lower()
            
            # Feature 1: URL complexity (length normalized)
            url_length = min(len(url) / 100.0, 1.0)
            
            # Feature 2: Suspicious patterns
            suspicious_patterns = sum([
                bool(re.search(r'\d+\.\d+\.\d+\.\d+', url)),
                any(tld in url_lower for tld in cls.SUSPICIOUS_TLDS),
                url.count('@') > 0,
                url.count('-') > 3,
            ]) / 4.0
            
            # Feature 3: Looks legitimate (has proper structure)
            has_protocol = url_lower.startswith('http')
            has_domain = bool(re.search(r'\.(com|org|net|edu|gov)', url_lower))
            looks_legit = (has_protocol and has_domain)
            
            # Feature 4: Structure complexity
            dot_count = min(url.count('.') / 5.0, 1.0)
            slash_count = min(url.count('/') / 10.0, 1.0)
            structure_complexity = (dot_count + slash_count) / 2.0
            
            return [
                url_length,
                suspicious_patterns,
                float(looks_legit),
                structure_complexity,
            ]
        except:
            return [0.0, 0.0, 0.0, 0.0]
    
    @classmethod
    def extract_text_quality_features(cls, text: str) -> List[float]:
        """Extract text quality indicators for gating"""
        if not text or pd.isna(text) or text == "":
            return [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        
        try:
            # Feature 1: Text length (normalized)
            text_length = min(len(text) / 500.0, 1.0)
            
            # Feature 2: Word count (normalized)
            words = text.split()
            word_count = min(len(words) / 50.0, 1.0)
            
            # Feature 3: Vocabulary richness
            unique_words = len(set(words))
            vocab_richness = unique_words / max(len(words), 1)
            
            # Feature 4: Special character density
            special_chars = len(re.findall(r'[^\w\s]', text))
            special_density = special_chars / max(len(text), 1)
            
            # Feature 5: Capitalization ratio
            capital_count = sum(1 for c in text if c.isupper())
            capital_ratio = capital_count / max(len(text), 1)
            
            # Feature 6: Coherence score (simple measure)
            avg_word_length = np.mean([len(w) for w in words]) if words else 0
            coherence = min(avg_word_length / 10.0, 1.0)
            
            return [
                text_length,
                word_count,
                vocab_richness,
                min(special_density, 0.5),
                min(capital_ratio, 0.5),
                coherence,
            ]
        except:
            return [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    
    @classmethod
    def extract_gating_features(cls, text: str, url: str, phrase_score: float) -> np.ndarray:
        """
        FIXED: Extract balanced features for gating network
        No artificial bias toward text expert
        """
        # Text quality features
        text_features = cls.extract_text_quality_features(text)
        
        # URL quality features
        url_features = cls.extract_url_quality_features(url)
        
        # Combined context features
        has_url = 1.0 if (url and not pd.isna(url) and url.strip()) else 0.0
        has_text = 1.0 if (text and not pd.isna(text) and text.strip()) else 0.0
        
        # Compute URL to text ratio
        url_text_ratio = 0.0
        if has_url and has_text:
            url_length = len(url) if url else 0
            text_length = len(text) if text else 0
            total_length = url_length + text_length
            url_text_ratio = url_length / total_length if total_length > 0 else 0.0
        
        # Combine all features
        features = np.array([
            has_url,                    # Binary: URL present
            has_text,                   # Binary: Text present
            phrase_score,               # Phishing phrase score
            url_text_ratio,             # URL-text length ratio
            *text_features,             # Text quality features (6)
            *url_features,              # URL quality features (4)
        ], dtype=np.float32)
        
        return features

# ============================================================================
# MODEL LOADER
# ============================================================================

class ModelLoader:
    """Centralized model loading with error handling"""
    
    @staticmethod
    def load_models(config: ModelConfig) -> Tuple:
        """Load all required models"""
        print("Loading Expert Models...")
        print("-" * 70)
        
        try:
            expert_1 = joblib.load(config.url_model_path)
            print("Expert 1 (URL-based): Loaded successfully")
        except Exception as e:
            print(f"Error loading URL expert: {e}")
            raise
        
        try:
            tokenizer = AutoTokenizer.from_pretrained(config.text_model_path)
            expert_2 = AutoModelForSequenceClassification.from_pretrained(config.text_model_path)
            expert_2.eval()
            print("Expert 2 (Text-based): Loaded successfully")
        except Exception as e:
            print(f"Error loading Text expert: {e}")
            raise
        
        try:
            # Use enhanced gating network with correct input size
            gating_net = EnhancedGatingNetwork(
                input_size=14,  # 2 binary + 1 phrase + 1 ratio + 6 text + 4 url = 14
                hidden_size=128,
                num_experts=2,
                dropout=0.3
            )
            gating_net.load_state_dict(torch.load(config.gating_network_path))
            gating_net.eval()
            print("Enhanced Gating Network: Loaded successfully")
        except Exception as e:
            print(f"Error loading Gating Network: {e}")
            print("Creating fallback gating network...")
            # Create simple fallback
            gating_net = EnhancedGatingNetwork(input_size=14, hidden_size=128, num_experts=2)
            gating_net.eval()
        
        print("\n" + "=" * 70)
        print("Enhanced MoE System Initialization Complete")
        print("=" * 70 + "\n")
        
        return expert_1, expert_2, tokenizer, gating_net

# ============================================================================
# BALANCED PHISHING DETECTOR
# ============================================================================

class BalancedPhishingDetector:
    """Main phishing detection system with balanced expert weighting"""
    
    def __init__(self, config: ModelConfig):
        self.config = config
        self.expert_1, self.expert_2, self.tokenizer, self.gating_net = \
            ModelLoader.load_models(config)
        self.feature_extractor = AdvancedFeatureExtractor()
        
        # For debugging and analysis
        self.prediction_history = []
    
    def predict(self, text: str, url: str = "") -> Dict:
        """
        Perform phishing detection with balanced expert weighting
        """
        # Preprocess inputs
        clean_text = self.feature_extractor.preprocess_text(text)
        phrase_score = self.feature_extractor.calculate_phrase_score(clean_text)
        
        # Determine input type
        has_url = bool(url and url.strip())
        has_text = bool(clean_text and clean_text.strip())
        
        # Get expert predictions
        url_probs = self._get_url_prediction(url)
        text_probs = self._get_text_prediction(clean_text)
        
        # Compute gating weights using enhanced network
        gating_features = self.feature_extractor.extract_gating_features(
            clean_text, url, phrase_score
        )
        expert_weights = self._compute_gating_weights(gating_features)
        
        # Weighted ensemble
        final_probs = (expert_weights[0] * url_probs + 
                      expert_weights[1] * text_probs)
        
        # Determine prediction
        prediction = "PHISHING" if final_probs[1] > self.config.phishing_threshold else "SAFE"
        confidence = max(final_probs) * 100
        
        # Expert predictions
        url_pred = "PHISHING" if url_probs[1] > 0.5 else "SAFE"
        text_pred = "PHISHING" if text_probs[1] > 0.5 else "SAFE"
        
        # Determine which expert dominates
        url_dominant = expert_weights[0] > expert_weights[1]
        
        # Determine routing method
        if has_url and not has_text:
            routing_method = "URL-only (No text available)"
            primary_expert = "URL Expert"
        elif has_text and not has_url:
            routing_method = "Text-only (No URL available)"
            primary_expert = "Text Expert"
        else:
            routing_method = "Balanced Gating Network"
            primary_expert = "URL Expert" if url_dominant else "Text Expert"
        
        # Store for analysis
        result = {
            'prediction': prediction,
            'confidence': confidence,
            'is_high_confidence': max(final_probs) > self.config.confidence_threshold,
            'url_weight': expert_weights[0] * 100,
            'text_weight': expert_weights[1] * 100,
            'url_prediction': url_pred,
            'text_prediction': text_pred,
            'url_confidence': max(url_probs) * 100,
            'text_confidence': max(text_probs) * 100,
            'phrase_score': phrase_score,
            'expert_agreement': url_pred == text_pred,
            'routing_method': routing_method,
            'primary_expert': primary_expert,
            'input_type': 'URL+Text' if (has_url and has_text) else ('URL' if has_url else 'Text'),
            'features': gating_features.tolist(),
            'url_dominant': url_dominant,
        }
        
        self.prediction_history.append(result)
        return result
    
    def _get_url_prediction(self, url: str) -> np.ndarray:
        """Get URL expert prediction with fallback"""
        if url and url.strip():
            try:
                url_df = pd.DataFrame({'url': [url]})
                return self.expert_1.predict_proba(url_df)[0]
            except Exception as e:
                print(f"URL expert error (using fallback): {e}")
        return np.array([0.5, 0.5])
    
    def _get_text_prediction(self, text: str) -> np.ndarray:
        """Get text expert prediction with fallback"""
        if text:
            try:
                inputs = self.tokenizer(
                    text, 
                    return_tensors='pt', 
                    padding=True,
                    truncation=True, 
                    max_length=self.config.max_text_length
                )
                with torch.no_grad():
                    outputs = self.expert_2(**inputs)
                    return torch.softmax(outputs.logits, dim=1)[0].numpy()
            except Exception as e:
                print(f"Text expert error (using fallback): {e}")
        return np.array([0.5, 0.5])
    
    def _compute_gating_weights(self, features: np.ndarray) -> np.ndarray:
        """Compute expert weights using gating network"""
        gating_input = torch.FloatTensor(features).unsqueeze(0)
        
        with torch.no_grad():
            weights = self.gating_net(gating_input)
        
        return weights[0].numpy()
    
    def analyze_weight_distribution(self, num_samples: int = 100) -> Dict:
        """Analyze weight distribution for debugging"""
        if len(self.prediction_history) < num_samples:
            samples = self.prediction_history
        else:
            samples = self.prediction_history[-num_samples:]
        
        url_weights = [s['url_weight'] for s in samples]
        text_weights = [s['text_weight'] for s in samples]
        
        return {
            'avg_url_weight': np.mean(url_weights),
            'avg_text_weight': np.mean(text_weights),
            'url_weight_std': np.std(url_weights),
            'text_weight_std': np.std(text_weights),
            'url_dominant_count': sum(1 for s in samples if s['url_dominant']),
            'text_dominant_count': sum(1 for s in samples if not s['url_dominant']),
            'url_dominant_pct': sum(1 for s in samples if s['url_dominant']) / len(samples) * 100,
        }

# ============================================================================
# TESTING INTERFACE
# ============================================================================

def display_enhanced_results(results: Dict, text: str = "", url: str = ""):
    """Display prediction results in formatted output"""
    print("=" * 80)
    print("BALANCED PREDICTION RESULTS - Enhanced MoE Phishing Detection")
    print("=" * 80)
    
    # Input display
    print(f"\nInput Type: {results['input_type']}")
    if text:
        display_text = text[:70] + "..." if len(text) > 70 else text
        print(f"Text: {display_text}")
    if url:
        print(f"URL: {url}")
    
    print("\n" + "-" * 80)
    
    # Gating analysis
    print("EXPERT WEIGHTING ANALYSIS:")
    print(f"  URL Expert Weight:  {results['url_weight']:6.2f}%")
    print(f"  Text Expert Weight: {results['text_weight']:6.2f}%")
    print(f"  Primary Expert:     {results['primary_expert']}")
    print(f"  Routing Method:     {results['routing_method']}")
    
    print("\nINDIVIDUAL EXPERT PREDICTIONS:")
    print(f"  URL Expert:  {results['url_prediction']:<10} "
          f"(Confidence: {results['url_confidence']:6.2f}%)")
    print(f"  Text Expert: {results['text_prediction']:<10} "
          f"(Confidence: {results['text_confidence']:6.2f}%)")
    
    print("\nADDITIONAL METRICS:")
    print(f"  Phrase Score:        {results['phrase_score']:6.3f}")
    print(f"  Expert Agreement:    {'Yes' if results['expert_agreement'] else 'No'}")
    
    print("-" * 80)
    
    # Final prediction
    confidence_level = "LOW" if results['confidence'] < 60 else \
                      "MODERATE" if results['confidence'] < 80 else "HIGH"
    
    prediction_symbol = "[PHISHING]" if results['prediction'] == 'PHISHING' else "[SAFE]"
    
    print(f"\nFINAL PREDICTION: {prediction_symbol} {results['prediction']}")
    print(f"Confidence: {results['confidence']:6.2f}% ({confidence_level})")
    print(f"High Confidence: {'Yes' if results['is_high_confidence'] else 'No'}")
    
    print("=" * 80)
    print()

def test_balanced_sample(detector: BalancedPhishingDetector, input_text: str) -> Dict:
    """Test the detector with automatic URL extraction"""
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    urls = re.findall(url_pattern, input_text)
    
    url = urls[0] if urls else ""
    text = re.sub(url_pattern, '', input_text).strip()
    
    results = detector.predict(text, url)
    display_enhanced_results(results, text, url)
    
    return results

def interactive_balanced_mode(detector: BalancedPhishingDetector):
    """Interactive testing interface"""
    print("\n" + "=" * 80)
    print("INTERACTIVE MODE - Balanced Phishing Detection")
    print("=" * 80)
    print("\nCommands:")
    print("  - Enter message/URL to analyze")
    print("  - 'exit' or 'quit' to end")
    print("  - 'sample' for predefined examples")
    print("  - 'batch' to test multiple samples")
    print("  - 'analyze' to view weight distribution")
    print("=" * 80)
    
    # Diverse test cases to demonstrate balanced weighting
    test_samples = [
        # URL-only cases (should favor URL expert)
        ("http://paypa1-security-login.com/verify", ""),
        ("", "https://amaz0n-account-update.xyz"),
        ("http://192.168.1.100/admin", ""),
        
        # Text-only cases (should favor text expert)
        ("URGENT! Your bank account has been suspended", ""),
        ("Congratulations! You won $5000 prize money", ""),
        ("Security alert: unusual login detected from new device", ""),
        
        # Balanced cases with both URL and text (gating should decide)
        ("Verify your PayPal account at http://paypal-secure-login.tk", ""),
        ("Claim your free iPhone: http://free-gift-claim.com Click now!", ""),
        ("Account update required: https://secure-update-bank.com", ""),
        
        # Ambiguous cases
        ("Meeting tomorrow at 3 PM", "http://teams.microsoft.com"),
        ("Your package was delivered", "https://fedex.com/track/123"),
        ("Weather alert for your area", "https://weather.com/alerts"),
        
        # Extreme cases
        ("", "http://very-long-suspicious-domain-name-with-many-hyphens-and-dots.xyz.tk"),
        ("URGENT! URGENT! URGENT! CLICK NOW! FREE MONEY! WINNER! PRIZE! SECURITY ALERT!", ""),
    ]
    
    while True:
        print("\n" + "-" * 80)
        user_input = input("\nEnter command or message: ").strip()
        
        if user_input.lower() in ['exit', 'quit', 'q']:
            print("\n" + "=" * 80)
            print("SESSION ENDED")
            print("=" * 80 + "\n")
            break
        
        if user_input.lower() == 'analyze':
            analysis = detector.analyze_weight_distribution()
            print("\nWEIGHT DISTRIBUTION ANALYSIS:")
            print(f"  Average URL Weight:  {analysis['avg_url_weight']:.2f}%")
            print(f"  Average Text Weight: {analysis['avg_text_weight']:.2f}%")
            print(f"  URL Dominant Cases:  {analysis['url_dominant_count']} "
                  f"({analysis['url_dominant_pct']:.1f}%)")
            print(f"  Text Dominant Cases: {analysis['text_dominant_count']}")
            continue
        
        if user_input.lower() == 'batch':
            print("\nTesting all samples...\n")
            for i, (sample_text, sample_url) in enumerate(test_samples, 1):
                print(f"\n{'='*80}")
                print(f"Sample {i}/{len(test_samples)}")
                print(f"{'='*80}")
                
                if sample_text and sample_url:
                    input_str = f"{sample_text} {sample_url}"
                elif sample_text:
                    input_str = sample_text
                else:
                    input_str = sample_url
                    
                test_balanced_sample(detector, input_str)
            continue
        
        if user_input.lower() == 'sample':
            print("\nPredefined Samples:")
            for i, (sample_text, sample_url) in enumerate(test_samples, 1):
                if sample_text and sample_url:
                    preview = f"{sample_text[:40]}... {sample_url[:30]}..."
                elif sample_text:
                    preview = sample_text[:70] + "..." if len(sample_text) > 70 else sample_text
                else:
                    preview = sample_url[:70] + "..." if len(sample_url) > 70 else sample_url
                print(f"{i:2d}. {preview}")
            
            try:
                choice = int(input(f"\nSelect sample (1-{len(test_samples)}): ")) - 1
                if 0 <= choice < len(test_samples):
                    sample_text, sample_url = test_samples[choice]
                    user_input = f"{sample_text} {sample_url}" if sample_text and sample_url else (sample_text or sample_url)
                else:
                    print("Invalid selection")
                    continue
            except ValueError:
                print("Invalid input")
                continue
        
        if not user_input:
            print("No input provided")
            continue
        
        try:
            test_balanced_sample(detector, user_input)
        except Exception as e:
            print(f"\nError: {str(e)}")

def demonstrate_balancing(detector: BalancedPhishingDetector):
    """Demonstrate the balancing mechanism"""
    print("\n" + "=" * 80)
    print("DEMONSTRATING BALANCED EXPERT WEIGHTING")
    print("=" * 80)
    
    demonstration_cases = [
        ("URL-only phishing", "", "http://paypal-login-secure.tk"),
        ("Text-only phishing", "URGENT! Verify account now!", ""),
        ("Mixed phishing", "Click here to claim prize", "http://free-prize-claim.xyz"),
        ("Legitimate with URL", "Meeting reminder", "https://teams.microsoft.com/meeting"),
        ("Legitimate text-only", "Weather update for tomorrow", ""),
    ]
    
    for case_name, text, url in demonstration_cases:
        print(f"\n{'='*80}")
        print(f"Case: {case_name}")
        print(f"{'='*80}")
        
        results = detector.predict(text, url)
        
        print(f"Input Type: {results['input_type']}")
        print(f"Weights - URL: {results['url_weight']:.1f}%, "
              f"Text: {results['text_weight']:.1f}%")
        print(f"Primary Expert: {results['primary_expert']}")
        print(f"Final Prediction: {results['prediction']} "
              f"({results['confidence']:.1f}% confidence)")
        
        # Check if weighting makes sense
        if not text and results['url_weight'] < 50:
            print("  Warning: URL-only input but text expert has higher weight!")
        elif not url and results['text_weight'] < 50:
            print("  Warning: Text-only input but URL expert has higher weight!")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    config = ModelConfig()
    detector = BalancedPhishingDetector(config)

Loading Expert Models...
----------------------------------------------------------------------
Expert 1 (URL-based): Loaded successfully
Expert 2 (Text-based): Loaded successfully
Error loading Gating Network: Error(s) in loading state_dict for EnhancedGatingNetwork:
	Missing key(s) in state_dict: "feature_processor.0.weight", "feature_processor.0.bias", "feature_processor.1.weight", "feature_processor.1.bias", "feature_processor.4.weight", "feature_processor.4.bias", "feature_processor.5.weight", "feature_processor.5.bias", "gate.0.weight", "gate.0.bias", "gate.2.weight", "gate.2.bias". 
	Unexpected key(s) in state_dict: "fc1.weight", "fc1.bias", "fc2.weight", "fc2.bias". 
Creating fallback gating network...

Enhanced MoE System Initialization Complete



In [ ]:
interactive_balanced_mode(detector)


INTERACTIVE MODE - Balanced Phishing Detection

Commands:
  - Enter message/URL to analyze
  - 'exit' or 'quit' to end
  - 'sample' for predefined examples
  - 'batch' to test multiple samples
  - 'analyze' to view weight distribution

--------------------------------------------------------------------------------



Enter command or message:  congratulations you can be part of 10 winners of 10000 gcash kindly register to the link to know more giveawaysgcassh.xyz


BALANCED PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: Text
Text: congratulations you can be part of 10 winners of 10000 gcash kindly re...

--------------------------------------------------------------------------------
EXPERT WEIGHTING ANALYSIS:
  URL Expert Weight:   40.85%
  Text Expert Weight:  59.15%
  Primary Expert:     Text Expert
  Routing Method:     Text-only (No URL available)

INDIVIDUAL EXPERT PREDICTIONS:
  URL Expert:  SAFE       (Confidence:  50.00%)
  Text Expert: PHISHING   (Confidence:  99.97%)

ADDITIONAL METRICS:
  Phrase Score:         0.308
  Expert Agreement:    No
--------------------------------------------------------------------------------

FINAL PREDICTION: [PHISHING] PHISHING
Confidence:  79.55% (MODERATE)
High Confidence: No


--------------------------------------------------------------------------------



Enter command or message:  https://sjhjjhsjjnm.wixsite.com/my-site-1


BALANCED PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: URL
URL: https://sjhjjhsjjnm.wixsite.com/my-site-1

--------------------------------------------------------------------------------
EXPERT WEIGHTING ANALYSIS:
  URL Expert Weight:   41.17%
  Text Expert Weight:  58.83%
  Primary Expert:     URL Expert
  Routing Method:     URL-only (No text available)

INDIVIDUAL EXPERT PREDICTIONS:
  URL Expert:  PHISHING   (Confidence:  98.75%)
  Text Expert: SAFE       (Confidence:  50.00%)

ADDITIONAL METRICS:
  Phrase Score:         0.000
  Expert Agreement:    No
--------------------------------------------------------------------------------

FINAL PREDICTION: [PHISHING] PHISHING
Confidence:  70.07% (MODERATE)
High Confidence: No


--------------------------------------------------------------------------------



Enter command or message:  Be part of 10 lucky winners of 3000 gcash just input your details at the given link https://sjhjjhsjjnm.wixsite.com/my-site-1


BALANCED PREDICTION RESULTS - Enhanced MoE Phishing Detection

Input Type: URL+Text
Text: Be part of 10 lucky winners of 3000 gcash just input your details at t...
URL: https://sjhjjhsjjnm.wixsite.com/my-site-1

--------------------------------------------------------------------------------
EXPERT WEIGHTING ANALYSIS:
  URL Expert Weight:   42.80%
  Text Expert Weight:  57.20%
  Primary Expert:     Text Expert
  Routing Method:     Balanced Gating Network

INDIVIDUAL EXPERT PREDICTIONS:
  URL Expert:  PHISHING   (Confidence:  98.75%)
  Text Expert: PHISHING   (Confidence:  91.82%)

ADDITIONAL METRICS:
  Phrase Score:         0.186
  Expert Agreement:    Yes
--------------------------------------------------------------------------------

FINAL PREDICTION: [PHISHING] PHISHING
Confidence:  94.79% (HIGH)
High Confidence: Yes


--------------------------------------------------------------------------------
